# 01 — Exploratory Data Analysis

This notebook presents a comprehensive exploratory data analysis (EDA) of the **IEEE-CIS Fraud Detection** dataset, originally released as part of the IEEE Computational Intelligence Society Kaggle competition.

The dataset contains real-world e-commerce transaction records provided by Vesta Corporation. Each transaction is labelled as fraudulent (`isFraud=1`) or legitimate (`isFraud=0`). The feature set includes:

- **Transaction features** — monetary amount, product code, card details, buyer/recipient email domains, and counting/matching features engineered by Vesta.
- **Identity features** — device type, device info, and digital fingerprint attributes linked via `TransactionID`.

The goal of this notebook is to understand the structure, distributions, missing-data patterns, and temporal dynamics of the data before building any predictive model. Insights gathered here will directly inform feature engineering and model design in subsequent notebooks.

---
**Project:** Pre-Fraud Detection Research (MSc Thesis)  
**Notebook:** 1 of N

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.feature_selection import mutual_info_classif

# Allow imports from the project root (one level up from /notebooks)
sys.path.insert(0, '..')

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.size": 11,
    "figure.dpi": 100,
})
warnings.filterwarnings("ignore")

# Colour palette used throughout the notebook
FRAUD_PALETTE = {0: "#3498db", 1: "#e74c3c"}
FRAUD_LABELS = {0: "Legitimate", 1: "Fraud"}

print("Imports loaded successfully.")

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
from src.data_loader import DataLoader

loader = DataLoader()
df = loader.load_ieee_data()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:20]} ... ({len(df.columns)} total)")
df.head()

## 1. Class Distribution

Fraud detection is inherently an **imbalanced classification** problem. Before any modelling we need to quantify the class imbalance so we can choose appropriate sampling strategies, evaluation metrics, and loss functions later.

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
class_counts = df["isFraud"].value_counts().sort_index()
class_pcts = df["isFraud"].value_counts(normalize=True).sort_index() * 100

print("Class Distribution")
print("=" * 40)
for label in class_counts.index:
    print(f"  {FRAUD_LABELS[label]:>12s} (isFraud={label}): "
          f"{class_counts[label]:>8,d}  ({class_pcts[label]:.2f}%)")
print(f"  {'Total':>12s}:              {len(df):>8,d}")
print(f"\n  Imbalance ratio (legit:fraud): {class_counts[0] / class_counts[1]:.1f} : 1")

# ── Bar chart ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
bars = axes[0].bar(
    [FRAUD_LABELS[i] for i in class_counts.index],
    class_counts.values,
    color=[FRAUD_PALETTE[i] for i in class_counts.index],
    edgecolor="black",
    linewidth=0.8,
)
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + len(df) * 0.01,
                 f"{count:,}", ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Transaction Counts by Class")
axes[0].set_ylabel("Number of Transactions")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Percentage
bars2 = axes[1].bar(
    [FRAUD_LABELS[i] for i in class_pcts.index],
    class_pcts.values,
    color=[FRAUD_PALETTE[i] for i in class_pcts.index],
    edgecolor="black",
    linewidth=0.8,
)
for bar, pct in zip(bars2, class_pcts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{pct:.2f}%", ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Transaction Percentage by Class")
axes[1].set_ylabel("Percentage (%)")

plt.tight_layout()
plt.savefig("class_distribution.png", bbox_inches="tight")  # saved to results later
plt.show()

## 2. Feature Distributions

Understanding how individual features differ between fraudulent and legitimate transactions helps identify which raw features already carry discriminative signal.

In [ ]:
# ── TransactionAmt distribution (log scale) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram: TransactionAmt by class (log-scaled x-axis)
for label in [0, 1]:
    subset = df[df["isFraud"] == label]["TransactionAmt"]
    axes[0].hist(
        np.log1p(subset),
        bins=100,
        alpha=0.6,
        label=FRAUD_LABELS[label],
        color=FRAUD_PALETTE[label],
        density=True,
    )
axes[0].set_title("Distribution of log(1 + TransactionAmt)")
axes[0].set_xlabel("log(1 + TransactionAmt)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Time-series: Fraud rate over TransactionDT
# TransactionDT is a timedelta in seconds from a reference point
dt_bins = pd.cut(df["TransactionDT"], bins=100)
fraud_rate_over_time = df.groupby(dt_bins, observed=False)["isFraud"].mean()

axes[1].plot(range(len(fraud_rate_over_time)), fraud_rate_over_time.values,
             color="#e74c3c", linewidth=1.2)
axes[1].fill_between(range(len(fraud_rate_over_time)), fraud_rate_over_time.values,
                      alpha=0.15, color="#e74c3c")
axes[1].set_title("Fraud Rate over TransactionDT (binned)")
axes[1].set_xlabel("Time Bin (chronological order)")
axes[1].set_ylabel("Fraud Rate")
axes[1].axhline(y=df["isFraud"].mean(), color="grey", linestyle="--", alpha=0.7,
                label=f"Overall rate: {df['isFraud'].mean():.3f}")
axes[1].legend()

plt.tight_layout()
plt.savefig("transaction_amt_and_time.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Correlation heatmap of top-30 features by variance ───────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "isFraud"]

# Select top 30 by variance (computed on non-null values)
variances = df[numeric_cols].var().dropna().sort_values(ascending=False)
top30 = variances.head(30).index.tolist()

corr_matrix = df[top30].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    annot=False,
    square=True,
    linewidths=0.3,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Correlation Heatmap — Top 30 Features by Variance", fontsize=15)
plt.tight_layout()
plt.savefig("correlation_heatmap_top30.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Box plots: fraud vs non-fraud for key numerical features ─────────────────
key_features = ["TransactionAmt", "D1", "D3", "C1", "C2"]
available_features = [f for f in key_features if f in df.columns]

fig, axes = plt.subplots(1, len(available_features), figsize=(4 * len(available_features), 6))
if len(available_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, available_features):
    data_plot = df[[feat, "isFraud"]].dropna()
    sns.boxplot(
        data=data_plot,
        x="isFraud",
        y=feat,
        palette=FRAUD_PALETTE,
        showfliers=False,  # hide outliers for readability
        ax=ax,
    )
    ax.set_xticklabels([FRAUD_LABELS[int(t.get_text())] for t in ax.get_xticklabels()])
    ax.set_title(feat)
    ax.set_xlabel("")

fig.suptitle("Box Plots — Key Features by Fraud Label (outliers hidden)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("boxplots_key_features.png", bbox_inches="tight")
plt.show()

## 3. Missing Data Analysis

Many features in the IEEE-CIS dataset contain substantial missing values. Missing data may not be random — it could itself be a signal of fraud (e.g., fraudsters providing fewer identity details). This section quantifies missingness and tests whether it correlates with the fraud label.

In [ ]:
# ── Missing data analysis ────────────────────────────────────────────────────

# 1. Overall missingness per feature
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct_nonzero = missing_pct[missing_pct > 0]

print(f"Features with any missing values: {len(missing_pct_nonzero)} / {len(df.columns)}")
print(f"Features with >50% missing:       {(missing_pct > 50).sum()}")
print(f"Features with >90% missing:       {(missing_pct > 90).sum()}")
print()

# List features with >50% missing
high_missing = missing_pct[missing_pct > 50]
print("Features with >50% missing values:")
print("-" * 45)
for feat, pct in high_missing.items():
    print(f"  {feat:<25s} {pct:6.2f}%")

# 2. Missing value heatmap (sample for performance)
sample_size = min(5000, len(df))
df_sample = df.sample(n=sample_size, random_state=42)

# Select columns that actually have missing data (up to 50 for readability)
cols_with_missing = missing_pct_nonzero.head(50).index.tolist()

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    df_sample[cols_with_missing].isnull().astype(int),
    cbar=False,
    yticklabels=False,
    cmap="YlOrRd",
    ax=ax,
)
ax.set_title(f"Missing Value Heatmap (top 50 features with missing data, {sample_size:,} sampled rows)")
ax.set_xlabel("Feature")
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig("missing_value_heatmap.png", bbox_inches="tight")
plt.show()

# 3. Missingness correlation with fraud label
print("\nCorrelation between missingness indicator and isFraud:")
print("=" * 55)
missingness_corr = {}
for col in df.columns:
    if df[col].isnull().any():
        corr = df[col].isnull().astype(int).corr(df["isFraud"])
        missingness_corr[col] = corr

missingness_corr_series = pd.Series(missingness_corr).sort_values(key=abs, ascending=False)
print(missingness_corr_series.head(20).to_string(float_format="{:.4f}".format))

## 4. Temporal Patterns

`TransactionDT` is a relative timestamp (seconds from a reference point). We derive hour-of-day and day-of-week proxies to look for cyclical fraud patterns — e.g., do fraudsters prefer certain hours or days?

In [ ]:
# ── Temporal patterns ────────────────────────────────────────────────────────

# Derive hour-of-day and day-of-week from TransactionDT (seconds from reference)
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["day_of_week"] = (df["TransactionDT"] // (3600 * 24)) % 7

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Fraud rate by hour of day
hourly = df.groupby("hour")["isFraud"].agg(["mean", "count"]).reset_index()
axes[0].bar(hourly["hour"], hourly["mean"], color="#e74c3c", alpha=0.75, edgecolor="black", linewidth=0.5)
axes[0].axhline(y=df["isFraud"].mean(), color="grey", linestyle="--", alpha=0.7, label="Overall rate")
axes[0].set_title("Fraud Rate by Hour of Day")
axes[0].set_xlabel("Hour (derived)")
axes[0].set_ylabel("Fraud Rate")
axes[0].set_xticks(range(0, 24))
axes[0].legend()

# 2. Fraud rate by day of week
daily = df.groupby("day_of_week")["isFraud"].agg(["mean", "count"]).reset_index()
day_names = ["Day 0", "Day 1", "Day 2", "Day 3", "Day 4", "Day 5", "Day 6"]
axes[1].bar(daily["day_of_week"], daily["mean"], color="#9b59b6", alpha=0.75, edgecolor="black", linewidth=0.5)
axes[1].axhline(y=df["isFraud"].mean(), color="grey", linestyle="--", alpha=0.7, label="Overall rate")
axes[1].set_title("Fraud Rate by Day of Week")
axes[1].set_xlabel("Day of Week (derived)")
axes[1].set_ylabel("Fraud Rate")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_names, rotation=45)
axes[1].legend()

# 3. Rolling average fraud rate (window = 5000 transactions)
df_sorted = df.sort_values("TransactionDT")
window_size = 5000
rolling_fraud = df_sorted["isFraud"].rolling(window=window_size, min_periods=1).mean()

axes[2].plot(range(len(rolling_fraud)), rolling_fraud.values, color="#e74c3c", linewidth=0.8, alpha=0.8)
axes[2].axhline(y=df["isFraud"].mean(), color="grey", linestyle="--", alpha=0.7, label="Overall rate")
axes[2].set_title(f"Rolling Fraud Rate (window = {window_size:,} transactions)")
axes[2].set_xlabel("Transaction Index (chronological)")
axes[2].set_ylabel("Fraud Rate")
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[2].legend()

plt.tight_layout()
plt.savefig("temporal_patterns.png", bbox_inches="tight")
plt.show()

# Clean up temporary columns
df.drop(columns=["hour", "day_of_week"], inplace=True, errors="ignore")

## 5. Feature Group Analysis

The IEEE-CIS dataset features can be grouped by their semantic role. We compute the **mutual information** (MI) of each feature with respect to the fraud label and then aggregate by group to understand which *families* of features are most informative.

| Group | Features | Description |
|-------|----------|-------------|
| **Direct** | TransactionAmt, ProductCD, card1–card6 | Transaction-level attributes |
| **Temporal** | TransactionDT, D1–D15 | Timedelta features |
| **Device** | DeviceType, DeviceInfo, id_* | Device fingerprinting |
| **Email** | P_emaildomain, R_emaildomain | Purchaser / recipient email |
| **Counting** | C1–C14 | Counting features (Vesta) |
| **Match** | M1–M9 | True/False match features |
| **Vesta** | V1–V339 | Vesta-engineered features |

In [ ]:
# ── Feature group analysis — Mutual Information ──────────────────────────────

def assign_group(col_name):
    """Assign a feature to its semantic group."""
    if col_name in ("TransactionAmt", "ProductCD") or col_name.startswith("card"):
        return "Direct"
    elif col_name == "TransactionDT" or (col_name.startswith("D") and col_name[1:].isdigit()):
        return "Temporal"
    elif col_name.startswith("id_") or col_name in ("DeviceType", "DeviceInfo"):
        return "Device"
    elif col_name in ("P_emaildomain", "R_emaildomain"):
        return "Email"
    elif col_name.startswith("C") and col_name[1:].isdigit():
        return "Counting"
    elif col_name.startswith("M") and col_name[1:].isdigit():
        return "Match"
    elif col_name.startswith("V") and col_name[1:].isdigit():
        return "Vesta"
    else:
        return "Other"

# Select only numeric features for MI computation
numeric_feats = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_feats = [c for c in numeric_feats if c != "isFraud"]

# For efficiency, fill NaN with -999 (MI requires no missing values)
X = df[numeric_feats].fillna(-999)
y = df["isFraud"]

print(f"Computing mutual information for {len(numeric_feats)} numeric features...")
mi_scores = mutual_info_classif(X, y, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({"feature": numeric_feats, "MI": mi_scores})
mi_df["group"] = mi_df["feature"].apply(assign_group)

print(f"Done. Top 10 features by MI:")
print(mi_df.nlargest(10, "MI")[["feature", "group", "MI"]].to_string(index=False))

# Aggregate: mean MI per group
group_mi = mi_df.groupby("group")["MI"].agg(["mean", "count", "sum"]).sort_values("mean", ascending=False)
group_mi.columns = ["Mean MI", "Feature Count", "Total MI"]
print("\nMean MI per Feature Group:")
print(group_mi.to_string())

# ── Bar chart ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mean MI per group
group_mi_sorted = group_mi.sort_values("Mean MI", ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(group_mi_sorted)))
axes[0].barh(group_mi_sorted.index, group_mi_sorted["Mean MI"], color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Mean Mutual Information per Feature Group")
axes[0].set_xlabel("Mean MI (nats)")

# Total MI per group
group_mi_sorted_total = group_mi.sort_values("Total MI", ascending=True)
axes[1].barh(group_mi_sorted_total.index, group_mi_sorted_total["Total MI"], color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_title("Total Mutual Information per Feature Group")
axes[1].set_xlabel("Total MI (nats)")

plt.tight_layout()
plt.savefig("feature_group_mi.png", bbox_inches="tight")
plt.show()

## Summary

**Key findings from this exploratory analysis:**

1. **Class imbalance** — _[placeholder: record the exact fraud percentage and imbalance ratio]_

2. **Transaction amount** — _[placeholder: note whether fraud transactions tend to be higher/lower in value]_

3. **Missing data** — _[placeholder: summarise how many features have >50% missing, and whether missingness correlates with fraud]_

4. **Temporal patterns** — _[placeholder: note any hour-of-day or day-of-week peaks in fraud rate]_

5. **Feature groups** — _[placeholder: which group had the highest mean MI? Which had the highest total MI?]_

6. **Correlated features** — _[placeholder: note any highly correlated feature pairs from the heatmap]_

---

**Next steps:**
- Notebook 02: Feature engineering and preprocessing pipeline
- Notebook 03: Baseline model training and evaluation

In [ ]:
# ── Save all figures to results/figures/eda/ ─────────────────────────────────
output_dir = os.path.join("..", "results", "figures", "eda")
os.makedirs(output_dir, exist_ok=True)

# List of figures generated during this notebook
figure_files = [
    "class_distribution.png",
    "transaction_amt_and_time.png",
    "correlation_heatmap_top30.png",
    "boxplots_key_features.png",
    "missing_value_heatmap.png",
    "temporal_patterns.png",
    "feature_group_mi.png",
]

import shutil

saved = []
for fig_name in figure_files:
    src = fig_name  # saved in notebooks/ (cwd) during execution
    dst = os.path.join(output_dir, fig_name)
    if os.path.exists(src):
        shutil.move(src, dst)
        saved.append(dst)
        print(f"  Saved: {dst}")
    else:
        print(f"  [SKIP] {src} not found (cell may not have been executed yet)")

print(f"\n{len(saved)} / {len(figure_files)} figures saved to {os.path.abspath(output_dir)}")